In [1]:
import os
import sys

# Add project root to path
nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)

Using project root: /root/Research/RithvikDecoder/Decoder


In [2]:
import numpy as np
import gymnasium as gym
from RL.gymnasium_env.envs.ldpc_decoder_env import LDPCDecoderEnv

## Create the Environment

The environment uses:
- The actual `BpDecoder` from ldpc
- The H matrix from `sims/H.mat`
- Same encoding and channel setup as in the sims notebooks

In [3]:
# Create environment with 6 clusters, max 30 iterations, SNR = 0 dB
env = LDPCDecoderEnv(num_clusters=6, max_iterations=30, snr_db=0)

print(f"Action space: {env.action_space}")
print(f"Observation space shape: {env.observation_space.shape}")
print(f"Number of clusters: {env.num_clusters}")

Action space: Discrete(6)
Observation space shape: (648,)
Number of clusters: 6


## Test with Random Actions

In [4]:
# Reset environment
observation, info = env.reset(seed=42)

print(f"Initial observation shape: {observation.shape}")
print(f"Initial LLR statistics:")
print(f"  Mean: {np.mean(observation):.4f}")
print(f"  Std:  {np.std(observation):.4f}")
print(f"  Min:  {np.min(observation):.4f}")
print(f"  Max:  {np.max(observation):.4f}")
print(f"\nInitial info: {info}")

Initial observation shape: (648,)
Initial LLR statistics:
  Mean: 0.2810
  Std:  4.8979
  Min:  -11.4607
  Max:  14.9043

Initial info: {'iteration': 0, 'syndrome_weight': 0}


In [5]:
# Run episode with random actions
total_reward = 0
step_count = 0

for step in range(30):
    # Sample random action
    action = env.action_space.sample()
    
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    if step < 5:  # Print first 5 steps
        print(f"Step {step+1}: Action={action}, Reward={reward:.1f}")
    
    if terminated:
        print(f"\nEpisode terminated at step {step+1}")
        break

print(f"\nResults:")
print(f"  Total reward: {total_reward}")
print(f"  Total steps: {step_count}")
print(f"  Converged: {info.get('is_converged', False)}")
print(f"  Decoded correctly: {info.get('decoded_correctly', False)}")

Step 1: Action=5, Reward=0.0
Step 2: Action=2, Reward=0.0
Step 3: Action=3, Reward=0.0
Step 4: Action=2, Reward=0.0
Step 5: Action=4, Reward=0.0

Episode terminated at step 30

Results:
  Total reward: 8.0
  Total steps: 30
  Converged: False
  Decoded correctly: False


## Test with Action 0 Only

Since the reward is 1 for action 0 and 0 for others, let's see what happens when we always choose action 0.

In [6]:
# Reset environment
observation, info = env.reset(seed=123)

total_reward = 0
step_count = 0

for step in range(30):
    # Always choose action 0
    action = 0
    
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    if step < 5:  # Print first 5 steps
        print(f"Step {step+1}: Action={action}, Reward={reward:.1f}")
    
    if terminated:
        print(f"\nEpisode terminated at step {step+1}")
        break

print(f"\nResults:")
print(f"  Total reward: {total_reward}")
print(f"  Total steps: {step_count}")
print(f"  Converged: {info.get('is_converged', False)}")
print(f"  Decoded correctly: {info.get('decoded_correctly', False)}")

Step 1: Action=0, Reward=1.0
Step 2: Action=0, Reward=1.0
Step 3: Action=0, Reward=1.0
Step 4: Action=0, Reward=1.0
Step 5: Action=0, Reward=1.0

Episode terminated at step 30

Results:
  Total reward: 30.0
  Total steps: 30
  Converged: False
  Decoded correctly: False


## Test at Different SNR Values

Let's test the environment at different SNR values to see how decoding performance varies.

In [7]:
snr_values = [-2, 0, 2, 4]
results = []

for snr in snr_values:
    env_snr = LDPCDecoderEnv(num_clusters=6, max_iterations=30, snr_db=snr)
    
    # Run 10 episodes
    successes = 0
    for episode in range(10):
        observation, info = env_snr.reset()
        
        for step in range(30):
            action = 0  # Always use action 0
            observation, reward, terminated, truncated, info = env_snr.step(action)
            
            if terminated:
                if info.get('decoded_correctly', False):
                    successes += 1
                break
    
    success_rate = successes / 10
    results.append((snr, success_rate))
    print(f"SNR = {snr:2d} dB: Success rate = {success_rate:.1%}")

print("\nDone!")

SNR = -2 dB: Success rate = 0.0%
SNR =  0 dB: Success rate = 0.0%
SNR =  2 dB: Success rate = 0.0%
SNR =  4 dB: Success rate = 0.0%

Done!


## Access Environment Components

The environment provides access to the underlying LDPC decoder components:

In [8]:
print("Environment components:")
print(f"  H matrix shape: {env.H.shape}")
print(f"  Number of check nodes (m): {env.m}")
print(f"  Codeword length (n_code): {env.n_code}")
print(f"  Message length (n): {env.n}")
print(f"  Cluster schedule shape: {env.schedule.shape}")
print(f"\nFirst cluster contains check nodes: {env.schedule[0][:10]}...")

Environment components:
  H matrix shape: (162, 648)
  Number of check nodes (m): 162
  Codeword length (n_code): 648
  Message length (n): 486
  Cluster schedule shape: (6, 27)

First cluster contains check nodes: [0 1 2 3 4 5 6 7 8 9]...
